# Prediction Routine

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.models import model_from_json  # type: ignore
from tensorflow.keras.preprocessing import image # type: ignore
from tensorflow.keras.preprocessing.image import ImageDataGenerator, smart_resize # type: ignore

from deep.modelling.custom_loss import CategoricalFocalLoss
from deep.predict.model_fetcher import fetch_model
from deep.predict.loss_calculator import get_loss
from deep.preprocess.image_cleaner import clean_test_directory 
from deep.preprocess.resize_inputs import resize_test_album 
from deep.predict.predict_album import predict_from_directory
from deep.constants import MODEL_DICT, MODEL_IMAGE_SIZE, CLEANER_JSONS, MODEL_CONFIGS, SPLITTER_JSONS, DATA_DIR, CLASS_LABELS

2025-05-01 23:30:35.055216: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 23:30:35.055949: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 23:30:35.059636: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 23:30:35.068935: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746138635.085932   21125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746138635.09

## Prediction Arguments

In [2]:
# Pipeline args
image_dir = DATA_DIR/ "TEST" # provide a valid directory with images
cleaner_config_path = CLEANER_JSONS / "cleaner_config_20250429T200053Z.json" # point to a valid cleaning json
split_config_path = SPLITTER_JSONS / "split_20250501T142850Z.json"

# Model initializing args
model_json_path = MODEL_CONFIGS / "model_base_architecture_2025-04-30.json"
model_name = input(f"Select a model from {list(MODEL_DICT.keys())}: ") # select one of our models

## Preprocessing 

This preprocessing schema adheres to the rule of no peeking, and modifies the provided data using only deterministic methods i.e. those learned from our training to show decent results.

In [ ]:
# Get list of image file paths
image_files = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]

# Cleaner
clean_test_directory(config_path=cleaner_config_path, input_dir=image_dir)

# Resizer
resize_test_album(model=MODEL_DICT[model_name], path=image_dir)

## Loading the Model

We check if the model is present in the `prediction_pipeline` directory, otherwise we fetch it - consult constants.py and the README.md for this module, for more details.

In [4]:
# Get Model weights
weights_path = fetch_model(model=model_name)

# Get Loss from Split info 
loss = get_loss(split_config_path)

# Load the model architecture from the JSON file
with model_json_path.open('r') as json_file:
    model_json = json_file.read()

custom_objects = {"CategoricalFocalLoss": CategoricalFocalLoss}
model = model_from_json(model_json, custom_objects={'CategoricalFocalLoss': loss})

# Load the model weights
model.load_weights(weights_path)

Model already exists at /home/nottoriousgg/MsC/DeepLearning/models/BASE_MODEL_00.weights.h5, skipping download.


2025-05-01 23:30:53.307852: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Prediction loop

Finally predictions are made on each image.

In [12]:
# Load class labels from constants
class_labels = CLASS_LABELS

# Set target image size
target_size = MODEL_IMAGE_SIZE[MODEL_DICT[model_name]]

# Initialize a list to store predictions
predictions = []

# Iterate through all images in the directory
for img_path in image_files:
    try:
        # Load and preprocess the image
        img = image.load_img(img_path, target_size=target_size)
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = smart_resize(img_array, target_size)
        img_array = img_array / 255.0

        preds = model.predict(img_array)

        class_idx = np.argmax(preds)

        predicted_label = class_labels[class_idx]

        # Append the label to the predictions list
        predictions.append({"filename": os.path.basename(img_path), "predicted_label": predicted_label})

    except Exception as e:
        print(f"Error processing {img_path}: {e}")

# Convert predictions to pandas DataFrame
df_preds = pd.DataFrame(predictions)

print(df_preds.head())


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step
               filename predicted_label
0  29455224_1048403.jpg      haliotidae
1  29455197_1048403.jpg      haliotidae
2  29455187_1048403.jpg      haliotidae
3  29455217_1048403.jpg      haliotidae
4  29455195_1048403.jpg      haliotidae


In [ ]:
# Store the results
df_preds.to_csv()

In [ ]:
df_preds
# trash :D 

#FIXME: IS likely suffering from issues with class label, consider its predictions with a grain of salt.

,filename,predicted_label
0,29455224_1048403.jpg,haliotidae
1,29455197_1048403.jpg,haliotidae
2,29455187_1048403.jpg,haliotidae
3,29455217_1048403.jpg,haliotidae
4,29455195_1048403.jpg,haliotidae
5,29455157_1048403.jpg,haliotidae
6,29455223_1048403.jpg,haliotidae
7,29455190_1048403.jpg,haliotidae
8,29455167_1048403.jpg,haliotidae
9,29455206_1048403.jpg,haliotidae
